# Homeowners Insurance Precision Pricing
## Actuarial Loss Modeling: GBM with Snowflake

This notebook walks through an end-to-end actuarial pricing workflow for a
**homeowners insurance** portfolio (~678,000 policies), using Snowflake as both the
data platform and the ML infrastructure.

**Modeling approach**

| Section | Model | Target | Snowflake Feature Used |
|---------|-------|--------|----------------------|
| 1–4 | Data prep & feature engineering | — | Snowpark (PySpark compat + native) |
| 5 | EDA | — | Snowpark DataFrames |
| 6 | Feature Store setup | — | Snowflake Feature Store |
| 7 | XGBoost GBM | Pure premium | Snowpark ML `XGBRegressor` |
| 8–9 | Model diagnostics | — | Experiment artifacts |
| 10 | Model registration | — | Snowflake Model Registry |
| 11 | Remote training | — | Snowflake ML Jobs |
| 12 | Batch scoring | — | Model Registry `run_batch` |
| 13 | Explainability | SHAP | Model Registry `explain` |

**Key actuarial concept:** The *pure premium* = expected total claim cost per unit of
exposure. It is the foundation of risk-based pricing and must be justified to state
regulators in a rate filing.

**Regulatory filing exhibits produced**
- Double-lift chart (Section 8) — primary model-validation exhibit for pricing committee review
- Lorenz / Gini curve (Section 9) — risk-discrimination summary for state rate filings

**Data source:** Two Snowflake tables in `COUNTRY_ML.ACTUARIAL_PRICING`:
`HOME_POLICY_FREQ` (~678K policy-level rows) and `HOME_POLICY_SEV` (~26K claim-level rows).
If using the XML ingestion path, this data is loaded from a `<PolicyFeed>` XML extract
(see `load_actuarial_data.py`).


In [ ]:
# Install Snowpark Connect for Apache Spark (required for Section 1)
# Run once and restart the kernel before proceeding.
# !pip install "snowpark-connect[jdk]"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import auc

# Snowpark Connect for Apache Spark — PySpark code runs on Snowflake compute
from snowflake.snowpark_connect import init_spark_session

sys.path.append("..")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

print("Libraries loaded.")
print(f"JAVA_HOME = {os.environ.get('JAVA_HOME', 'NOT SET — see .env file')}")

In [ ]:
# ── Demo configuration ─────────────────────────────────────────────────────
# Edit config.py to change environment values — this cell imports from there.
# All other scripts (load_actuarial_data.py, train.py) read from config.py too.
from src.config import DATABASE, SCHEMA, ROLE, WAREHOUSE, COMPUTE_POOL

## 1. Load Data from Snowflake via Snowpark Connect

The source data lives in two Snowflake tables:

| Table | Rows | Granularity |
|-------|------|-------------|
| `HOME_POLICY_FREQ` | ~678,000 | One row per policy — exposure, claim count, and rating factors |
| `HOME_POLICY_SEV` | ~26,000 | One row per **claim** — multiple rows per policy if > 1 claim occurred |

**Why two tables?** Frequency and severity are modelled separately in actuarial practice.
A Poisson model predicts *how many* claims occur; a Gamma model predicts the *average size*
of each claim given that one occurred. The pure premium (their product) is the final pricing
output. The tables are joined by `POLICY_ID` to produce a policy-level dataset for modelling.

### Snowpark Connect for Apache Spark

This section uses **Snowpark Connect for Apache Spark**, which executes standard PySpark
DataFrame code on Snowflake's compute engine — no local Spark cluster required.

**Migration note:** Snowpark Connect is a compatibility bridge for teams with existing
PySpark codebases. It does **not** produce Snowflake Horizon lineage, which is required for
column-level audit trails in actuarial regulatory submissions. See Section 4 for the
equivalent native Snowpark implementation that provides full lineage.


**Query history note:** Snowpark Connect queries appear in Snowflake query history as standard warehouse queries. There is no separate Spark UI — use **Query Profile** (`Monitoring → Query History`) to inspect execution plans, scan statistics, and per-step warehouse credit usage.

In [ ]:

try:
    # From SPCS Service, session is already configured
    from snowflake.snowpark.context import get_active_session
    snowpark_session = get_active_session()
except Exception:
    # Local IDE, supply session parameters
    from snowflake.snowpark import Session
    snowpark_session = Session.builder.configs({
        "connection_name": "default",
        "role":      ROLE,
        "warehouse": WAREHOUSE,
        "database":  DATABASE,
        "schema":    SCHEMA,
    }).create()

snowpark_session.sql(f"USE ROLE {ROLE}").collect()
snowpark_session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()
snowpark_session.sql(f"USE DATABASE {DATABASE}").collect()
snowpark_session.sql(f"USE SCHEMA {SCHEMA}").collect()

spark = init_spark_session() # Replaces getOrCreate

## 2. Data Cleaning & Derived Actuarial Targets

### Outlier capping (actuarial standard practice)

| Field | Cap | Rationale |
|-------|-----|-----------|
| `CLAIM_COUNT` | 4 | Policies with 5+ claims are likely data errors; extreme counts distort frequency model fit |
| `EXPOSURE` | 1.0 year | Mid-term policies spanning calendar years can exceed 1.0; cap to one full policy-year |
| `CLAIM_AMOUNT` | $200,000 | Catastrophic (CAT) losses are handled by reinsurance and excluded from attritional pricing |

These caps mirror standard GLM data conditioning applied before fitting frequency and
severity components in property/casualty actuarial workflows.

### Missing value imputation

Median imputation is applied to four continuous rating factors. Median is preferred over
mean for actuarial data because loss distributions are right-skewed — a handful of large
losses would shift the mean away from the typical policy experience.

### Derived actuarial targets

| Target | Formula | Model role |
|--------|---------|-----------|
| `PURE_PREMIUM` | `CLAIM_AMOUNT / EXPOSURE` | Primary pricing target — the risk cost per policy-year |
| `FREQUENCY` | `CLAIM_COUNT / EXPOSURE` | Step 1 of the two-stage GLM (Poisson distribution) |
| `AVG_CLAIM_AMOUNT` | `CLAIM_AMOUNT / max(CLAIM_COUNT, 1)` | Step 2 of the two-stage GLM (Gamma distribution) |

The **pure premium** is the actuarial north star: it represents the expected risk cost
that must be recovered in base rates before loading for expenses, profit, and contingencies.

Actuarial rating factors require domain-specific encoding choices:

| Rating Factor | Encoding | Reason |
|---|---|---|
| `PROPERTY_AGE`, `POLICYHOLDER_AGE` | Quantile bins (10 bins) | Non-linear relationship with loss — age effects plateau and reverse at extremes |
| `CONSTRUCTION_TYPE`, `CONSTRUCTION_QUALITY`, `OCCUPANCY_TYPE`, `REGION_CODE`, `TERRITORY_CODE` | One-hot encoding | Nominal / ordinal categorical — each level has an independent relativty in GLMs |
| `LOSS_HISTORY_SCORE` | Passthrough | Already on a meaningful numeric scale (experience modifier, ~100 = neutral) |
| `POPULATION_DENSITY` | Log + standard scale | Right-skewed across several orders of magnitude; log transform stabilises variance |

### Two implementation options

The same feature engineering pipeline is demonstrated twice:
1. **Snowpark Connect (PySpark API)** — runs existing PySpark code on Snowflake compute; no lineage
2. **Native Snowpark (recommended)** — full column-level lineage through Snowflake Horizon; required for regulatory audit trails

#### Option 1: Snowpark Connect (Spark API)


In [ ]:
from pyspark.sql import functions as F

# Read from Snowflake table (within Database, Schema set previously)
freq_sdf = spark.read.table("HOME_POLICY_FREQ")

# Joins and other manipulation push compute to Snowflake warehouse
sev_sdf  = (
    spark.read.table("HOME_POLICY_SEV")
    .groupBy("POLICY_ID")
    .agg(F.sum("CLAIM_AMOUNT").alias("CLAIM_AMOUNT"))
)

policy_sdf = (
    freq_sdf
    .join(sev_sdf, on="POLICY_ID", how="left")
    .withColumn("CLAIM_AMOUNT", F.coalesce(F.col("CLAIM_AMOUNT"), F.lit(0.0)))
)

policy_sdf.printSchema()

In [ ]:
# --- Clip outliers in Spark -----------------------------------------------
policy_sdf = (
    policy_sdf
    .withColumn("CLAIM_COUNT",  F.least(F.col("CLAIM_COUNT"),  F.lit(4)))
    .withColumn("EXPOSURE",     F.least(F.col("EXPOSURE"),     F.lit(1.0)))
    .withColumn("CLAIM_AMOUNT", F.least(F.col("CLAIM_AMOUNT"), F.lit(200_000.0)))
)

# Consistency: zero-amount policies should have zero claim count
policy_sdf = policy_sdf.withColumn(
    "CLAIM_COUNT",
    F.when(
        (F.col("CLAIM_AMOUNT") == 0) & (F.col("CLAIM_COUNT") >= 1),
        F.lit(0)
    ).otherwise(F.col("CLAIM_COUNT"))
)

# --- Median imputation via approxQuantile (no MLlib) ----------------------
numeric_cols = ["LOSS_HISTORY_SCORE", "POPULATION_DENSITY", "PROPERTY_AGE", "POLICYHOLDER_AGE"]
medians = policy_sdf.approxQuantile(numeric_cols, [0.5], relativeError=0.001)

for col, med in zip(numeric_cols, medians):
    policy_sdf = policy_sdf.withColumn(
        f"{col}_CLEAN",
        F.coalesce(F.col(col), F.lit(med[0]))
    )

policy_sdf.show(5)

In [ ]:
# ── 1. Quantile binning: PROPERTY_AGE_CLEAN, POLICYHOLDER_AGE_CLEAN ──────────
bin_cols = ["PROPERTY_AGE_CLEAN", "POLICYHOLDER_AGE_CLEAN"]
quantile_points = [i / 10 for i in range(1, 10)]  # 9 cut points → 10 bins

binned_feature_cols = []
for col_name in bin_cols:
    boundaries = policy_sdf.approxQuantile(col_name, quantile_points, 0.001)

    # Assign bin index (iterate high → low so first match wins)
    bin_expr = F.lit(len(boundaries))
    for i in range(len(boundaries) - 1, -1, -1):
        bin_expr = F.when(F.col(col_name) <= F.lit(boundaries[i]), F.lit(i)).otherwise(bin_expr)
    policy_sdf = policy_sdf.withColumn(f"{col_name}_BIN", bin_expr)

    # One-hot expand bin index into indicator columns
    for b in range(len(boundaries) + 1):
        feat_col = f"{col_name}_BIN_{b}"
        policy_sdf = policy_sdf.withColumn(
            feat_col,
            F.when(F.col(f"{col_name}_BIN") == b, F.lit(1)).otherwise(F.lit(0)).cast("double")
        )
        binned_feature_cols.append(feat_col)

# ── 2. One-hot encode categoricals ───────────────────────────────────────────
cat_cols = [
    "CONSTRUCTION_TYPE", "CONSTRUCTION_QUALITY",
    "OCCUPANCY_TYPE", "REGION_CODE", "TERRITORY_CODE",
]
ohe_feature_cols = []
for col_name in cat_cols:
    distinct_vals = sorted([
        row[0] for row in policy_sdf.select(col_name).distinct().collect()
        if row[0] is not None
    ])
    for val in distinct_vals:
        safe_val = str(val).upper().replace(" ", "_").replace("-", "_")
        feat_col = f"{col_name}_{safe_val}"
        policy_sdf = policy_sdf.withColumn(
            feat_col,
            F.when(F.col(col_name) == F.lit(val), F.lit(1)).otherwise(F.lit(0)).cast("double")
        )
        ohe_feature_cols.append(feat_col)

# ── 3. Log-standardize POPULATION_DENSITY_CLEAN ──────────────────────────────
log_stats = policy_sdf.agg(
    F.mean(F.log(F.col("POPULATION_DENSITY_CLEAN"))).alias("log_mean"),
    F.stddev(F.log(F.col("POPULATION_DENSITY_CLEAN"))).alias("log_std"),
).collect()[0]

policy_sdf = policy_sdf.withColumn(
    "POPULATION_DENSITY_LOG_SCALED",
    (F.log(F.col("POPULATION_DENSITY_CLEAN")) - F.lit(log_stats["log_mean"]))
    / F.lit(log_stats["log_std"])
)

# ── 4. Assemble feature column list ─────────────────────────────────────────
feature_cols = (
    binned_feature_cols
    + ohe_feature_cols
    + ["LOSS_HISTORY_SCORE_CLEAN", "POPULATION_DENSITY_LOG_SCALED"]
)
policy_sdf.select(["POLICY_ID"] + feature_cols + ["CLAIM_AMOUNT"]) \
    .write.mode("overwrite").saveAsTable(f"{SCHEMA}.ML_INPUT_SPARK")


### Option 2: Migrate to Native Snowpark

**Pros**

* Full Snowflake lineage — every `with_column`, `join`, and `save_as_table` is a first-class Snowflake object; column-level lineage flows end-to-end through Horizon for regulatory audit trails
* No JVM — no Java dependency, no gRPC size limits, no local Spark server
* Native Snowpark ML — direct access to the Feature Store, Model Registry, Snowflake Experiments, and batch scoring via `.run()` without round-tripping through pandas
* Warehouse cost visibility — every transformation runs as a Snowflake query with full query profile, credit attribution, and warehouse-level governance
* Streamlit in Snowflake — model outputs and actuarial diagnostics (double-lift, Gini) can be served directly as SiS apps against the feature table with no data egress
* Access control + data governance — Dynamic Data Masking, row-level security, and Snowflake Horizon apply natively to Snowpark DataFrames and output tables
* Simpler deployment — Snowpark Python runs in SPCS or Snowflake Notebooks without any local environment setup

**Cons**

* One-time migration cost for existing Spark-heavy codebases
* Engineers must learn Snowpark-specific idioms (e.g., `F.ln` vs `F.log`, join deduplication behavior)
* Some advanced Spark MLlib algorithms have no direct Snowpark equivalent and require `sklearn` or Snowpark ML alternatives


### Snowpark Migration Skill

Use the built-in `spark-migration` skill to accelerate this development. Feel free to try it out and replace the next cell with what you get!

> @actuarial_pricing_demo.ipynb Carefully examine the first few cells of this notebook and convert them from Spark APIs to Snowpark APIs. Include all steps from reading from the source tables, performing feature engineering, and saving the engineered features to a new table.  Explain any API differences in comments. Add this as a new cell in the notebook. Do not touch any other cells in the notebook.

In [ ]:
# ============================================================
# Snowpark Native API: Data Loading + Feature Engineering
# ============================================================
#
# Replicates cells 3–6 (Snowpark Connect / PySpark) using the
# native Snowpark Python API, then writes the feature matrix to
# a Snowflake table — avoiding the gRPC 128 MB transfer limit
# that causes cell 6 to fail on .toPandas() with this dataset.
#
# API differences (PySpark  →  Snowpark Python)
# ─────────────────────────────────────────────
#  spark.read.table("T")               session.table("T")
#  from pyspark.sql import functions   from snowflake.snowpark import functions
#  df.withColumn("c", expr)            df.with_column("c", expr)   [snake_case]
#  df.groupBy(...)                     df.group_by(...)            [snake_case]
#  df.toPandas()                       df.to_pandas()
#  df.approxQuantile(cols,[0.5],err)   F.approx_percentile(col, 0.5) aggregate
#  F.log(x)  ← natural log in PySpark  F.ln(x)  ← Snowflake LOG() is base-10 ⚠️
#  df.write.saveAsTable("T")           df.write.save_as_table("T") [snake_case]
# ============================================================

from snowflake.snowpark import functions as F

# Reuse the Snowpark session from cell 2 — no Spark cluster needed.
session = snowpark_session

# ── 1. Load and join source tables ────────────────────────────────────────────
# session.table() replaces spark.read.table()
freq_df = session.table("HOME_POLICY_FREQ")

# group_by() is the Snowpark snake_case equivalent of PySpark groupBy()
sev_agg_df = (
    session.table("HOME_POLICY_SEV")
    .group_by("POLICY_ID")
    .agg(F.sum("CLAIM_AMOUNT").alias("CLAIM_AMOUNT"))
)

# join() with on="col_name" (string) deduplicates the join key, matching PySpark.
# HOME_POLICY_FREQ has no CLAIM_AMOUNT column, so there is no column conflict.
policy_df = (
    freq_df
    .join(sev_agg_df, on="POLICY_ID", how="left")
    # with_column() replaces PySpark withColumn() [snake_case]
    .with_column("CLAIM_AMOUNT", F.coalesce(F.col("CLAIM_AMOUNT"), F.lit(0.0)))
)

# ── 2. Clip outliers ──────────────────────────────────────────────────
policy_df = (
    policy_df
    .with_column("CLAIM_COUNT",  F.least(F.col("CLAIM_COUNT"),  F.lit(4)))
    .with_column("EXPOSURE",     F.least(F.col("EXPOSURE"),     F.lit(1.0)))
    .with_column("CLAIM_AMOUNT", F.least(F.col("CLAIM_AMOUNT"), F.lit(200_000.0)))
    .with_column(
        "CLAIM_COUNT",
        F.when(
            (F.col("CLAIM_AMOUNT") == 0) & (F.col("CLAIM_COUNT") >= 1),
            F.lit(0)
        ).otherwise(F.col("CLAIM_COUNT"))
    )
)

# ── 3. Median imputation ───────────────────────────────────────────────
# PySpark: df.approxQuantile(cols, [0.5], relativeError) is a DataFrame method.
# Snowpark: F.approx_percentile(col, p) is an aggregate — gather all medians
#           in one server-side query rather than pulling data to the client.
numeric_cols = ["LOSS_HISTORY_SCORE", "POPULATION_DENSITY", "PROPERTY_AGE", "POLICYHOLDER_AGE"]
median_row = policy_df.select([
    F.approx_percentile(F.col(c), 0.5).alias(f"{c}_MEDIAN") for c in numeric_cols
]).collect()[0]

for c in numeric_cols:
    policy_df = policy_df.with_column(
        f"{c}_CLEAN",
        F.coalesce(F.col(c), F.lit(float(median_row[f"{c}_MEDIAN"])))
    )

# ── 4. Quantile binning ────────────────────────────────────────────────
bin_cols = ["PROPERTY_AGE_CLEAN", "POLICYHOLDER_AGE_CLEAN"]
quantile_points = [i / 10 for i in range(1, 10)]  # 9 cut points → 10 bins

binned_feature_cols_sp = []
for col_name in bin_cols:
    # Compute all 9 boundaries in one server-side query
    boundaries_row = policy_df.select([
        F.approx_percentile(F.col(col_name), p).alias(f"P{idx}")
        for idx, p in enumerate(quantile_points, start=1)
    ]).collect()[0]
    boundaries = [float(boundaries_row[f"P{idx}"]) for idx in range(1, len(quantile_points) + 1)]

    bin_expr = F.lit(len(boundaries))
    for i in range(len(boundaries) - 1, -1, -1):
        bin_expr = F.when(F.col(col_name) <= F.lit(boundaries[i]), F.lit(i)).otherwise(bin_expr)
    policy_df = policy_df.with_column(f"{col_name}_BIN", bin_expr)

    for b in range(len(boundaries) + 1):
        feat_col = f"{col_name}_BIN_{b}"
        policy_df = policy_df.with_column(
            feat_col,
            F.when(F.col(f"{col_name}_BIN") == b, F.lit(1)).otherwise(F.lit(0)).cast("double")
        )
        binned_feature_cols_sp.append(feat_col)

# ── 5. One-hot encode categoricals ─────────────────────────────────────────
# Row field access is identical: row[0] or row["col_name"] both work in Snowpark.
cat_cols = [
    "CONSTRUCTION_TYPE", "CONSTRUCTION_QUALITY",
    "OCCUPANCY_TYPE", "REGION_CODE", "TERRITORY_CODE",
]
ohe_feature_cols_sp = []
for col_name in cat_cols:
    distinct_vals = sorted([
        row[0] for row in policy_df.select(col_name).distinct().collect()
        if row[0] is not None
    ])
    for val in distinct_vals:
        safe_val = str(val).upper().replace(" ", "_").replace("-", "_")
        feat_col = f"{col_name}_{safe_val}"
        policy_df = policy_df.with_column(
            feat_col,
            F.when(F.col(col_name) == F.lit(val), F.lit(1)).otherwise(F.lit(0)).cast("double")
        )
        ohe_feature_cols_sp.append(feat_col)

# ── 6. Log-standardize POPULATION_DENSITY_CLEAN ────────────────────────────
# KEY DIFFERENCE: PySpark F.log(x) is the natural logarithm.
#                 Snowflake/Snowpark F.log(base, x) requires a base argument;
#                 LOG(x) alone is base-10 in Snowflake SQL.
#                 → Use F.ln(x) for the natural logarithm in Snowpark.
log_stats = policy_df.select(
    F.mean(F.ln(F.col("POPULATION_DENSITY_CLEAN"))).alias("LOG_MEAN"),
    F.stddev(F.ln(F.col("POPULATION_DENSITY_CLEAN"))).alias("LOG_STD"),
).collect()[0]

policy_df = policy_df.with_column(
    "POPULATION_DENSITY_LOG_SCALED",
    (F.ln(F.col("POPULATION_DENSITY_CLEAN")) - F.lit(float(log_stats["LOG_MEAN"])))
    / F.lit(float(log_stats["LOG_STD"]))
)

# ── 7. Save feature matrix to Snowflake ────────────────────────────────────
# Instead of .to_pandas() (which would hit the same 128 MB gRPC limit),
# write the result directly to Snowflake. Downstream cells can reload it with
# session.table("HOME_POLICY_FEATURES").to_pandas() in manageable chunks,
# or use it directly with Snowpark ML / Model Registry.
#
# save_as_table() replaces PySpark df.write.saveAsTable() [snake_case]
feature_cols_sp = (
    binned_feature_cols_sp
    + ohe_feature_cols_sp
    + ["LOSS_HISTORY_SCORE_CLEAN", "POPULATION_DENSITY_LOG_SCALED"]
)
output_cols = ["POLICY_ID", "CLAIM_COUNT", "EXPOSURE", "CLAIM_AMOUNT"] + feature_cols_sp

OUTPUT_TABLE = "ML_INPUT_SNOWFLAKE"
(
    policy_df.select(output_cols)
    .write
    .mode("overwrite")
    .save_as_table(OUTPUT_TABLE)
)

print(f"Saved to {OUTPUT_TABLE}: {len(feature_cols_sp)} engineered feature columns")
print(f"  Reload: session.table('{OUTPUT_TABLE}').to_pandas()")

## 5. Exploratory Data Analysis

Before fitting any model, examine how portfolio exposure is distributed across the key
rating factors. This step:

- **Identifies thin segments** — a rating factor level with little exposure may produce
  unreliable model estimates; credibility weighting or grouping may be required
- **Detects data quality issues** — unexpected distribution spikes often reveal ingestion problems
- **Informs binning choices** — quantile bin boundaries in Section 4 are driven by the
  actual portfolio distribution, not arbitrary round numbers

All charts are **exposure-weighted** (policy-years) rather than policy-count-weighted,
matching how actuaries analyse portfolio mix. A policy on cover for 0.1 years should
contribute 10× less than a full-year policy.


In [ ]:
df = policy_df.to_pandas() # Read from our non-materialzied Snowpark DataFrame previously
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (col, label) in zip(axes, [
    ("POLICYHOLDER_AGE",  "Policyholder Age"),
    ("PROPERTY_AGE",      "Property Age (years)"),
    ("LOSS_HISTORY_SCORE","Loss History Score"),
]):
    ax.hist(df[col], bins=min(30, df[col].nunique()),
            weights=df["EXPOSURE"], color="steelblue", alpha=0.75, edgecolor="none")
    ax.set_xlabel(label)
    ax.set_ylabel("Total Exposure (policy-years)")
    ax.set_title(f"{label} Distribution")

plt.suptitle("Portfolio Exposure Distribution by Key Rating Factors", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

### Hands-On

Materialise the Snowpark DataFrame to pandas and experiment with different
segmentations. Try plotting `TERRITORY_CODE` or `CONSTRUCTION_TYPE` distribution
to see how the portfolio is concentrated across rating territories.


## 6. Snowflake Feature Store & Training Datasets

The Snowflake **Feature Store** provides versioned, reproducible feature sets —
a core requirement for actuarial model governance and regulatory audit trails.

| Capability | Why it matters for pricing models |
|---|---|
| **Versioned feature views** | Re-run a historical filing analysis with the exact feature definitions used at the original submission date |
| **Dataset snapshots** | Freeze the training/validation split so any future re-run produces identical results |
| **Entity-based joins** | New policies can be scored against the same feature logic without re-engineering |
| **Label separation** | The Feature Store holds all policies; train/test splits happen at dataset-generation time, not at feature-engineering time |

### Workflow in this section

1. **Feature view** (`ACTUARIAL_FEATURES`) — wraps `ML_INPUT_SNOWFLAKE` in a versioned
   Feature Store object. Increment `version` when upstream feature engineering changes.
2. **Training dataset** (`ACTUARIAL_TRAINING`) — joins the feature view onto a *spine* of
   training-split `POLICY_ID`s and their labels. Only training policies are included.
3. **Validation dataset** (`ACTUARIAL_VALIDATION`) — same for the held-out validation split.
   The split is frozen at generation time, making it reproducible across all future training runs.

> **Docs:** [Snowflake Feature Store](https://docs.snowflake.com/en/developer-guide/snowflake-ml/feature-store/overview)


In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode

# ── 1. Load table and add target column ──────────────────────────────────────
ml_sdf = (
        snowpark_session.table("ML_INPUT_SNOWFLAKE")
        .with_column("PURE_PREMIUM", F.col("CLAIM_AMOUNT") / F.col("EXPOSURE"))
    )

# ── 3. Split (operates on Snowpark DataFrame — no data transfer) ──────────────
train_sdf, test_sdf = ml_sdf.random_split([0.75, 0.25], seed=0)

# ── 3. Entity ─────────────────────────────────────────────────────────────────
fs = FeatureStore(
    session=snowpark_session,
    database=DATABASE,
    name=SCHEMA,
    default_warehouse=WAREHOUSE,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

policy_entity = Entity(name="POLICY", join_keys=["POLICY_ID"])
try:
    fs.register_entity(policy_entity)
except Exception:
    pass # Already exists

# ── 4. Feature view: all engineered features across all policies ──────────────
# Source is ml_sdf (full population, not the training split).
# Feature Store holds features for every entity; the train/test cut
# happens at dataset generation time via the spine.
fv = FeatureView(
    name="ACTUARIAL_FEATURES",
    entities=[policy_entity],
    feature_df=ml_sdf.select(["POLICY_ID"] + feature_cols), # THIS IS THE KEY LINE - FILL Feature View with data we engineered
    refresh_freq=None,
    desc="Engineered insurance rating facotrs. "
)
fv = fs.register_feature_view(
    feature_view=fv,
    version="1", # Make some changes, and update this!
    block=True,
    overwrite=True,
)

# ── 5. Training dataset: features joined onto the training spine ──────────────
# Only training-split POLICY_IDs are included; test_sdf stays withheld.
spine_df = train_sdf.select("POLICY_ID", "PURE_PREMIUM", "EXPOSURE")

training_dataset = fs.generate_dataset(
    name="ACTUARIAL_TRAINING",
    spine_df=spine_df,
    features=[fv],
    version="3", # <-- Update the data split, then increment this
    spine_label_cols=["PURE_PREMIUM", "EXPOSURE"],
    desc="Training split",
)

spine_df = test_sdf.select("POLICY_ID", "PURE_PREMIUM", "EXPOSURE")

validation_dataset = fs.generate_dataset(
    name="ACTUARIAL_VALIDATION",
    spine_df=spine_df,
    features=[fv],
    version="3", # <-- Update the data split, then increment this
    spine_label_cols=["PURE_PREMIUM", "EXPOSURE"],
    desc="Validation split",
)

## 7. GBM Training (Snowflake-Managed XGBoost)

Gradient Boosting Machines capture non-linear interactions between rating factors that
GLMs cannot model — making GBMs a strong actuarial pricing benchmark and an increasingly
common primary model in property/casualty personal lines.

**Why GBM for pure premium?**
- Handles complex interactions (e.g. construction type × region × age) without manual feature crosses
- Produces competitive Gini indices on held-out validation data
- SHAP values (Section 13) provide the per-policy attribution needed for rate justification

### `snowflake.ml.modeling.xgboost.XGBRegressor`

Snowflake's managed XGBRegressor wraps the open-source XGBoost library with a Snowpark
DataFrame interface:
- No `.to_pandas()` call — training data never leaves Snowflake
- `input_cols` / `label_cols` / `sample_weight_col` replace sklearn's positional `X`, `y`
  convention, making column selection explicit and audit-friendly
- `sample_weight_col="EXPOSURE"` is critical: policies with partial-year exposure contribute
  proportionally to the loss function — without this, short-term policies are over-represented

Evaluation metrics (RMSE, MAE) are computed as Snowpark SQL aggregations — no validation
rows are transferred to the client for scoring.

> **Docs:** [Snowpark ML Modeling](https://docs.snowflake.com/en/developer-guide/snowflake-ml/modeling/overview)


In [ ]:
from snowflake.ml.modeling.xgboost import XGBRegressor

train_ds_sdf = training_dataset.read.to_snowpark_dataframe()
test_ds_sdf = training_dataset.read.to_snowpark_dataframe()

_spine_cols = {"POLICY_ID", "PURE_PREMIUM", "EXPOSURE"}
feature_cols = [c for c in train_ds_sdf.columns if c not in _spine_cols]


# ── 3. Fit (training runs in the Snowflake warehouse) ─────────────────────────
gbm_sp = XGBRegressor(
        input_cols=feature_cols,
        label_cols=["PURE_PREMIUM"],
        output_cols=["PREDICTED_PURE_PREMIUM"],
        sample_weight_col="EXPOSURE",
        n_estimators=200,
        learning_rate=0.05,
        max_leaves=31,
        random_state=0,
        objective="reg:squarederror",
    )
gbm_sp.fit(train_ds_sdf)

    # ── 4. Predict + score (server-side aggregation) ──────────────────────────────
preds_sdf = gbm_sp.predict(test_ds_sdf)

metrics = preds_sdf.select(
        F.sqrt(F.avg(
            (F.col("PURE_PREMIUM") - F.col("PREDICTED_PURE_PREMIUM")) ** 2
        )).alias("RMSE"),
        F.avg(F.abs(
            F.col("PURE_PREMIUM") - F.col("PREDICTED_PURE_PREMIUM"))
        ).alias("MAE"),
    ).collect()[0]

print(f"Test RMSE : {metrics['RMSE']:>10.4f}")
print(f"Test MAE  : {metrics['MAE']:>10.4f}")

## 7. Double-Lift Chart

The double-lift chart is a standard actuarial model validation tool:

1. Score all test policies with each model
2. Rank into 10 deciles (1 = lowest predicted risk, 10 = highest)
3. For each decile: compare exposure-weighted **predicted** vs. **observed** pure premium

**What to look for:**
- Both lines should rise monotonically from left (safest) to right (riskiest)
- Predicted and observed lines should track closely — a large gap means poor calibration in that risk segment
- The steeper the slope, the better the model discriminates risk

In [ ]:
# y_pred_product = y_pred_product_test   # statsmodels Freq×Sev predictions
# y_pred_tweedie = glm_pp.predict(X_test)
from src.helpers import double_lift_chart

df_test = gbm_sp.predict(test_sdf).to_pandas()
df_test["PurePremium"] = df_test["PURE_PREMIUM"]

y_pred_gbm     = df_test["PREDICTED_PURE_PREMIUM"].values

double_lift_chart(
    df_test,
    predictions_dict={
        "GBM":          y_pred_gbm,
    },
    n_bins=10,
)
plt.show()

## 8. Lorenz Curve & Gini Index

The Lorenz curve plots cumulative claim amounts against cumulative exposure,
where policies are ordered from **safest** (lowest predicted risk) to **riskiest**.

- **Random baseline** (diagonal): no discrimination ability
- **Oracle model** (upper bound): policies ranked by their actual observed losses
- **Our models**: the farther above the diagonal, the better the risk discrimination

The **Gini index** = area between the Lorenz curve and the diagonal (normalized to [0,1]).
Higher is better — it measures how well the model separates low-risk from high-risk policies.

> This chart is one of the primary deliverables actuaries produce for pricing reviews
> and regulatory model documentation.

In [ ]:
from src.helpers import lorenz_curve

fig, ax = plt.subplots(figsize=(8, 8))

cum_exp, cum_claims = lorenz_curve(
        df_test["PurePremium"], y_pred_gbm, df_test["EXPOSURE"]
)
gini = 1 - 2 * auc(cum_exp, cum_claims)
ax.plot(cum_exp, cum_claims, label=f"{label}  (Gini = {gini:.3f})")

# Oracle upper bound
cum_exp, cum_claims = lorenz_curve(
    df_test["PurePremium"], df_test["PurePremium"], df_test["EXPOSURE"]
)
gini = 1 - 2 * auc(cum_exp, cum_claims)
ax.plot(cum_exp, cum_claims, linestyle="-.", color="gray",
        label=f"Oracle  (Gini = {gini:.3f})")

# Random baseline
ax.plot([0, 1], [0, 1], linestyle="--", color="black", label="Random baseline")

ax.set(
    title="Lorenz Curves — Homeowners Pure Premium Ranking",
    xlabel="Cumulative proportion of exposure\n(ordered safest → riskiest by model)",
    ylabel="Cumulative proportion of claim amounts",
)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 10. Model Registry

The Snowflake **Model Registry** stores fitted models as versioned, governed objects in
Snowflake — alongside their metrics, signatures, and lineage back to the training data.

| Capability | Value for actuarial teams |
|---|---|
| **Versioned model objects** | Compare RMSE / Gini across model versions; promote a specific version for rate implementation |
| **Auto-generated input signature** | Enforces the correct feature schema at scoring time — prevents silent schema drift |
| **Linked to experiment run** | Every model version links back to the Snowflake Experiment run that produced it |
| **`enable_explainability=True`** | Pre-builds SHAP infrastructure so `mv.run(..., function_name="explain")` works without re-training |
| **`target_platforms`** | Controls where the model can be served: `WAREHOUSE` for inline batch scoring, `SNOWPARK_CONTAINER_SERVICES` for `run_batch` on a compute pool |

> **Docs:** [Snowflake Model Registry](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)


In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model.task import Task

registry = Registry(
    session=snowpark_session,
    database_name=DATABASE,
    schema_name=SCHEMA,
)

mv = registry.log_model(
    model=gbm_sp,
    model_name="ACTUARIAL_GBM",
    version_name="V3",
    task=Task.TABULAR_REGRESSION,
    comment="XGBoost pure premium model — homeowners actuarial pricing demo",
    metrics={
        "test_rmse": float(metrics["RMSE"]),
        "test_mae": float(metrics["MAE"])
    },
    sample_input_data=train_ds_sdf.limit(10),
)

## 11. Remote Training with Snowflake ML Jobs

As models and datasets scale, training inside a notebook becomes impractical:
notebooks block the UI, can be interrupted mid-run, and don't support concurrent
experiments across team members.

**Snowflake ML Jobs** submit training as an asynchronous containerised job to a
**Compute Pool** — a managed cluster of CPU or GPU instances in Snowflake's SPCS.

### Key design points

- `submit_file("train.py", compute_pool=COMPUTE_POOL, query_warehouse=WAREHOUSE, ...)` uploads
  the script and launches the job; `job.wait()` blocks until it finishes
- The model is registered directly in the Model Registry **from inside the container** — no
  data or model artefacts leave Snowflake
- `SNOWFLAKE_SERVICE_NAME` is auto-injected into the container by SPCS and used as the model
  version name — every job submission produces a unique, traceable version with no manual bookkeeping
- `query_warehouse` must be passed at submission time; `USE WAREHOUSE` is not permitted from
  inside the container

### `train.py`

`train.py` is the production training entrypoint. It is a self-contained file (no config.py
import) that runs the full pipeline: dataset load → hyperparameter logging → XGBoost fit →
evaluation → diagnostic plots → model registration.

> **Docs:** [Snowflake ML Jobs](https://docs.snowflake.com/en/developer-guide/snowflake-ml/ml-jobs/overview)


In [ ]:
from snowflake.ml.jobs import submit_file

job = submit_file(
    # Use CoCo or see https://docs.snowflake.com/en/developer-guide/snowpark-ml/reference/latest/api/jobs/snowflake.ml.jobs.submit_file for the signature!
    "../src/train.py",
    compute_pool=COMPUTE_POOL,
    stage_name="payload_stage",
    query_warehouse=WAREHOUSE,
    args=["--ds-version=1"],
    session=session,
)
print(f"Submitted job {job.name}")
job.wait()
print(job.status)
print(job.get_logs())

## 12. Batch Inference & Export

After training, the most common actuarial scoring pattern is **batch inference**:
score the entire validation portfolio, then load predictions into Snowflake for
downstream analysis and export to external actuarial tools.

### `mv.run_batch()` vs `mv.run()`

| | `mv.run()` | `mv.run_batch()` |
|---|---|---|
| Execution | Warehouse (inline, synchronous) | Compute Pool (async ML Job) |
| Best for | Quick spot-checks, small datasets | Full portfolio scoring (700K+ policies) |
| Returns | Snowpark DataFrame | `MLJob` → Parquet files written to a stage |

### Workflow

1. `load_dataset(...).read.to_snowpark_dataframe(only_feature_cols=True)` — loads validation
   features, automatically dropping label columns (`PURE_PREMIUM`, `EXPOSURE`) so only
   model inputs are sent to inference
2. `mv.run_batch(...)` — scores all policies on the compute pool and writes predictions to
   `OUTPUT_STAGE`
3. Results are loaded into a Snowflake table via `COPY INTO`, then exported to a stage as XML
   (via `TO_XML` + Snowflake's XML file format) for downstream actuarial tools

> **Docs:** [Batch inference](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)


In [ ]:
from snowflake.ml.dataset import load_dataset
from snowflake.ml.model._client.model.batch_inference_specs import JobSpec, OutputSpec, SaveMode

validation_features_spdf = load_dataset(session, name="ACTUARIAL_VALIDATION", version="2").read.to_snowpark_dataframe(only_feature_cols=True)

registry = Registry(session=snowpark_session, database_name=DATABASE, schema_name=SCHEMA)
mv = registry.get_model("ACTUARIAL_GBM").default


batch_job = mv.run_batch(
    validation_features_spdf,
    compute_pool=COMPUTE_POOL,
    output_spec=OutputSpec(
        stage_location=f"@{DATABASE}.{SCHEMA}.OUTPUT_STAGE/ACTUARIAL_GBM/",
        mode=SaveMode.OVERWRITE,
    ),
    job_spec=JobSpec(
        replicas=1,
        function_name="PREDICT",
    ),
)

print(f"Batch job submitted: {batch_job.name}")
batch_job.wait()
print(f"Status: {batch_job.status}")
print(batch_job.get_logs())

In [ ]:
# Read batch inference predictions back from stage
# Results are written as Parquet by run_batch; load them into a Snowpark DataFrame for inspection.

pred_sdf = snowpark_session.read.parquet(
    f"@{DATABASE}.{SCHEMA}.OUTPUT_STAGE/ACTUARIAL_GBM/"
)
pred_sdf.show(10)

In [ ]:
# Query ML Job logs from the Snowflake event table
# Requires an active event table configured at the account level.
# See: https://docs.snowflake.com/en/developer-guide/logging-tracing/logging

job_logs_query = f"""
    SELECT
        TIMESTAMP,
        RESOURCE_ATTRIBUTES['snow.job.name']::STRING  AS job_name,
        RECORD_ATTRIBUTES['log.iostream']::STRING     AS stream,
        VALUE:body::STRING                            AS message
    FROM SNOWFLAKE.TELEMETRY.EVENTS
    WHERE RESOURCE_ATTRIBUTES['snow.job.name']::STRING = '{batch_job.name}'
      AND TIMESTAMP > DATEADD('hour', -2, CURRENT_TIMESTAMP())
    ORDER BY TIMESTAMP
"""

snowpark_session.sql(job_logs_query).show(50)

## 13. Model Explainability with SHAP

Regulatory model documentation increasingly requires **feature-level explanations** —
per-policy attribution showing why each policy received its predicted premium.

Snowflake's Model Registry supports SHAP out of the box when a model is registered with
`options={"enable_explainability": True}` (set in `train.py`).

| Output | Description |
|--------|-------------|
| `mv.run(df, function_name="explain")` | Returns a DataFrame with a `<FEATURE>_explanation` column per rating factor |
| `plot_force(...)` | Force plot for a single policy showing which factors pushed the predicted premium up or down from the baseline |

**Actuarial use cases for per-policy SHAP values:**
- **Rate justification:** explain to regulators why a specific risk tier commands a higher premium
- **Agent/broker communication:** provide interpretable premium breakdowns at point of sale
- **Adverse action notices:** document the top rating factors contributing to a pricing decision
- **Model governance:** confirm that the model's top factors align with actuarial expectations


In [ ]:
import altair as alt
from snowflake.ml.monitoring.explain_visualize import plot_force

# Required for Vega/Altair charts to render in Snowflake Notebooks
alt.renderers.enable("mimetype")

mv = registry.get_model("ACTUARIAL_GBM").default
explanations = mv.run(validation_features_spdf, function_name="EXPLAIN")

explanations_pd = explanations.to_pandas()
explanations_pd = explanations_pd.drop(columns=[c for c in explanations_pd if "_explanation" not in c])
validation_features_pd = validation_features_spdf.to_pandas()
plot_force(explanations_pd.iloc[10], validation_features_pd.drop(columns=["POLICY_ID"]).iloc[10])